# 31_error_analysis_CHS

- 목적: 모델 노트북(21·24·25·26)의 `metrics.csv`를 모아 비교표 `results/model_comparison.csv`를 만들고, 미탐(FN)·오경보(FP)가 몰리는 조건과 그 원인(상관 구조 변화, 정상 부분공간 이탈, 분포 차이)을 정리한다.
- 입력: `results/2*/metrics.csv`, `results/2*/error_cases.csv`, `results/04_data_profile_CHS/normal_suspects.csv`, `results/03_diagnosis_recheck_CHS/sine_fit_by_segment.csv`(세그먼트 속성), `data/processed/11_window_features.parquet`(D·E·F, C 일부)
- 출력: `results/model_comparison.csv`, `figures/31_error_analysis_CHS/`, `results/31_error_analysis_CHS/` (`fn_by_model.csv`, `fn_category_summary.csv`, `fn_check_vs_metrics.csv`, `fp_by_state.csv`, `fp_by_block.csv`, `fp_common_segments.csv`, `corr_break_normal_vs_outlier.csv`, `fn_pca_spe_t2.csv`, `ks_segment_normal_vs_outlier.csv`)
- 담당: CHS
- 심사 대응: 2번(모델 비교표), 3번(FN/FP 집중 조건), 4번(상태별 경보 운영 근거), 5번(라벨 노이즈·DC 누수 관점의 FN 해석)

원칙: 이상 이벤트는 1건(1초 기준 평가 가능 세그먼트 17개)이므로 FN/FP 결과는 모집단 추론이 아니라 이 이벤트의 기술이다. 표준화·PCA·임계는 **train fold 정상**으로만 적합한다. 검정 p값은 세그먼트 대표값에서만 계산한다(`stats` 규칙). 상태 번호는 parquet·03·04 규약(**0 = 고부하, 1 = 저부하**)을 따른다. 21 노트북의 규칙 내부 상태 번호(0 = 저부하)와 반대이므로 섞지 않는다.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from scipy import stats as sst
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "31_error_analysis_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
_fonts = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ("Malgun Gothic", "AppleGothic") if f in _fonts), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
import evaluate, features, split, stats
from sklearn.decomposition import PCA
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
HAS_PQ = PQ.exists()
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT), "| font:", plt.rcParams["font.family"], "| parquet:", HAS_PQ)

## 0. 공통 헬퍼 (src 이관 후보)

- `load_metrics` · `fn_long` · `seg_attrs` → `src/evaluate.py`(또는 신규 `src/analysis.py`) 이관 후보. `corr_break` · `pca_spe_t2` → `src/stats.py` 이관 후보.
- FN 범주: **판정 불가**(0·5·9·10, 1초 윈도우 없음 — 평가 분모에서 빠짐) / **전 채널 조용**(3·19·20, 표준 전처리 후 진동·전류 모두 정상 범위, 19·20은 원시 DC만 이상, 21 V2-a) / **기타**.

In [ ]:
UNDECIDABLE = (0, 5, 9, 10)            # 길이 < 10샘플 → 1초 윈도우 없음
QUIET = (3, 19, 20)                    # 표준 전처리 후 전 채널 조용 (19·20: 원시 DC만 이상)
STATE_NAME = {0: "고부하", 1: "저부하"}  # parquet·03·04·benchmark_jiw 규약
OPT_COLS = ["win_s", "n_out_seg", "n_detected"]


def load_metrics(results_dir=paths.RESULTS) -> pd.DataFrame:
    """results/*/metrics.csv(스모크 제외)를 합쳐 notebook·model_base·family 열을 붙인다."""
    frames = []
    for p in sorted(results_dir.glob("*/metrics.csv")):
        m = pd.read_csv(p, dtype={"fold": str})
        m.insert(0, "notebook", p.parent.name)
        frames.append(m[["notebook"] + evaluate.METRIC_COLS + [c for c in OPT_COLS if c in m.columns]])
    M = pd.concat(frames, ignore_index=True)
    M["model_base"] = M.model.str.replace(r"_w\d+s$", "", regex=True)       # 24~26은 모델명에 _w1s 접미사
    suf = M.model.str.extract(r"_w(\d+)s$")[0].astype(float)
    assert (suf.isna() | (suf == M.win_s)).all(), "모델명 접미사와 win_s 불일치"
    M["family"] = M.notebook.str.extract(r"^\d+_model_([a-z]+)_")[0]
    return M


def fn_category(seg_id: int) -> str:
    return "판정 불가" if seg_id in UNDECIDABLE else ("전 채널 조용" if seg_id in QUIET else "기타")


def fn_long(results_dir=paths.RESULTS) -> pd.DataFrame:
    """노트북별 error_cases.csv를 (notebook, model, win_s, split, seg_uid, n_miss) 긴 표로 통일한다.

    - 21: 이상 세그먼트 1행 × det_<model> 열(group_kfold_seg 1초 OOF, NaN = 윈도우 없음) → n_miss 0/1
    - 24~26: type=FN 행(첫 시드 기준) → (model, split, seg_uid)별 미탐 fold 수. FN 0건 모델은 행이 없다
    """
    rows = []
    for p in sorted(results_dir.glob("2*/error_cases.csv")):
        e = pd.read_csv(p)
        if "type" not in e.columns:                                          # 21 wide 형식
            for c in [c for c in e.columns if c.startswith("det_")]:
                d = e.loc[e[c].notna(), ["seg_uid"]].assign(model=c[4:], win_s=1.0, split="group_kfold_seg",
                                                            n_miss=(~e.loc[e[c].notna(), c].astype(bool)).astype(int))
                rows.append(d.assign(notebook=p.parent.name))
            continue
        ids = pd.read_csv(p.parent / "metrics.csv")[["model", "win_s"]].drop_duplicates()
        fn = (e[e.type == "FN"].groupby(["model_id", "split", "seg_uid"]).size().rename("n_miss").reset_index()
              .rename(columns={"model_id": "model"}).merge(ids, on="model", how="left"))
        rows.append(fn.assign(notebook=p.parent.name))
    L = pd.concat(rows, ignore_index=True)
    L["seg_id"] = L.seg_uid.str.extract(r"_(\d+)$")[0].astype(int)
    L["fn_category"] = L.seg_id.map(fn_category)
    return L[["notebook", "model", "win_s", "split", "seg_uid", "seg_id", "fn_category", "n_miss"]]


def seg_attrs(W=None) -> pd.DataFrame:
    """세그먼트 1행 속성(seg_uid, seg_id, src, len, state, block, vib_grade, cur_grade).

    parquet이 없으면 03 sine_fit_by_segment.csv(620 세그먼트, state·len·grade 포함)로 대신하고,
    block은 split.block_assignment를 seg_id 순서(= 시간순, 04 normal_suspects.block과 일치)로 재현한다.
    """
    s = (pd.read_csv(paths.RESULTS / "03_diagnosis_recheck_CHS" / "sine_fit_by_segment.csv")
         .drop_duplicates("seg_uid")[["seg_uid", "seg_id", "src", "len", "state", "vib_grade", "cur_grade"]])
    nrm = s[s.src == "normal"]
    s["block"] = s.seg_uid.map(split.block_assignment(nrm, n_blocks=5, start_col="seg_id"))
    if W is not None:                                                        # parquet 값이 있으면 그쪽을 우선
        wb = W.groupby("seg_uid")[["state", "block"]].first()
        s["state"] = s.seg_uid.map(wb.state).fillna(s.state)
        s["block"] = s.seg_uid.map(wb.block).fillna(s.block)
    s["state_name"] = s.state.map(STATE_NAME)
    return s.reset_index(drop=True)


def corr_break(a: pd.DataFrame, b: pd.DataFrame, cols, method: str = "spearman") -> pd.DataFrame:
    """두 표본(a=정상, b=이상)의 쌍별 상관과 차이 |rho_a − rho_b| 긴 표 + (Ca, Cb)."""
    Ca, Cb = a[cols].corr(method=method), b[cols].corr(method=method)
    iu = np.triu_indices(len(cols), 1)
    out = pd.DataFrame({"feature_a": np.array(cols)[iu[0]], "feature_b": np.array(cols)[iu[1]],
                        "rho_normal": Ca.to_numpy()[iu], "rho_outlier": Cb.to_numpy()[iu]})
    out["diff"] = out.rho_outlier - out.rho_normal
    out["abs_diff"] = out["diff"].abs()
    return out.sort_values("abs_diff", ascending=False).reset_index(drop=True), Ca, Cb


def pca_spe_t2(train: pd.DataFrame, X: pd.DataFrame, cols, var: float = 0.95) -> tuple[pd.DataFrame, int]:
    """train 정상으로 StandardScaler+PCA(누적 분산 var 성분 수 k) 적합 → X의 SPE(Q)·Hotelling T².

    SPE = 표준화 공간 재구성 제곱합 = len(cols) × stats.pca_recon_error²,
    T² = Σ_{i≤k} score_i² / λ_i. nan 행은 nan.
    """
    k = stats.pca_fit_report(train, cols, var_targets=(var,))["n_for"][var]
    rep = stats.pca_fit_report(train, cols, n_components=k)
    x = X[cols].to_numpy(dtype=float)
    ok = ~np.isnan(x).any(axis=1)
    spe = len(cols) * stats.pca_recon_error(rep, x) ** 2
    t2 = np.full(len(x), np.nan)
    z = rep["scaler"].transform(x[ok])
    t2[ok] = ((rep["pca"].transform(z) ** 2) / rep["pca"].explained_variance_).sum(axis=1)
    return pd.DataFrame({"spe": spe, "t2": t2}, index=X.index), k

## A. 모델 비교표 `model_comparison.csv`

- 목적: 심사 2번(40점) 최종 비교표. 모든 모델이 같은 입력·분할·지표로 평가됐는지 확인하고 fold 평균 행만 모은다.
- 방법: `results/*/metrics.csv`(11의 `metrics_smoke.csv`는 파일명이 달라 자동 제외)를 합치고 `notebook` 열을 붙인다. 공통 7열 + `win_s`·`n_out_seg`·`n_detected`만 남기고 `fold == "mean"` 행을 `paths.RESULTS / "model_comparison.csv"`로 저장한다(직접 편집 금지, 착수 가이드 §1). `detect_rate = n_detected / n_out_seg`를 추가한다.
- 노트북 간 규약 차이(확인 결과):
  - 모델명: 21은 `rule1_rms_ai0`처럼 윈도우 접미사가 없고 `win_s` 열로 구분, 24~26은 `cnn_deepant_w1s`처럼 접미사 포함 → `model_base`(접미사 제거) 열로 통일, 접미사와 `win_s` 일치를 assert.
  - 윈도우: 21은 1·2·3초, 24~26은 1·2초만 → 3초 비교는 규칙 4종뿐.
  - 시드: 24·25(딥러닝)와 26의 `deep_svdd`는 3시드 평균이라 `n_detected`가 소수(예: 16.33)가 될 수 있다. 26의 표 모델(MCD·OC-SVM·HBOS·COPOD)은 1시드.
  - 피처군 ablation: 26만 `_amp`/`_amp_sine`/`_amp_shape`/`_amp_rel` 접미사로 피처군을 구분한다. **`_sine`·`_rel`은 세그먼트 브로드캐스트 값(날짜 교란 가능성)** 이므로 기본(진폭) 성능과 분리해 읽는다(착수 가이드 §7).
  - mean 행의 `n_out_seg`·`n_detected`는 fold 평균(group_kfold_seg 3.4 = 17/5)이다.

In [ ]:
M_ALL = load_metrics()
print("metrics 행:", len(M_ALL), M_ALL.groupby("notebook").size().to_dict())
CMP = M_ALL[M_ALL.fold == "mean"].drop(columns="fold").copy()
CMP["detect_rate"] = CMP.n_detected / CMP.n_out_seg
CMP = CMP.sort_values(["split", "win_s", "auc"], ascending=[True, True, False]).reset_index(drop=True)
CMP.to_csv(paths.RESULTS / "model_comparison.csv", index=False)
print("model_comparison.csv 행:", len(CMP), "| (split, win_s)별:", CMP.groupby(["split", "win_s"]).size().to_dict())
SHOW = ["notebook", "model_base", "auc", "fpr_sample", "fpr_segment", "delay_median_s", "detect_rate"]
for sp in ("group_kfold_seg", "time_block"):
    print(f"\n[{sp}, 1초] AUC 순")
    print(CMP[(CMP.split == sp) & (CMP.win_s == 1)][SHOW].round(4).to_string(index=False))

In [ ]:
# AUC vs 세그먼트 오경보율 (group_kfold_seg, 1초) — 오른쪽 위가 좋은 것이 아니라 "AUC 높고 fpr_segment 낮은" 왼쪽 위가 좋다
FAM_COL = {"rule": "#52514e", "forecasting": "#2a78d6", "graph": "#1baf7a", "distribution": "#eb6834"}
c1 = CMP[(CMP.split == "group_kfold_seg") & (CMP.win_s == 1)]
fig, ax = plt.subplots(figsize=(8, 5.5))
for fam, g in c1.groupby("family"):
    ax.scatter(g.fpr_segment, g.auc, s=40, color=FAM_COL.get(fam, "k"), label=fam, edgecolor="white", lw=1.5)
for r in c1.nlargest(4, "auc").itertuples():
    ax.annotate(r.model_base, (r.fpr_segment, r.auc), fontsize=7, xytext=(4, -8), textcoords="offset points")
for r in c1[c1.family == "rule"].itertuples():
    ax.annotate(r.model_base, (r.fpr_segment, r.auc), fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.set_xlabel("fpr_segment (정상 세그먼트 오경보율)"); ax.set_ylabel("AUC (fold 평균)")
ax.set_title("모델 비교 — group_kfold_seg, 1초 윈도우"); ax.legend(title="계열", fontsize=8); ax.grid(alpha=.3)
paths.save_fig(FIG, "model_auc_vs_fpr_segment.png")

## B. FN 이원화 — 모델별 미탐 이상 세그먼트

- 목적: 심사 3번. 미탐이 "모델 한계"인지 "라벨 노이즈(전 채널 조용)"인지 나눈다. 21 규칙 4종은 3·19·20을 모두 놓쳤다(21 §V2-a).
- 방법: 노트북별 `error_cases.csv`를 `fn_long`으로 통일(21 = 세그먼트 × `det_*` wide 형식, 1초 group_kfold_seg OOF만 / 24~26 = `type`=FP·FN 긴 형식, **첫 시드만**, 두 분할·1·2초). FN을 판정 불가·전 채널 조용·기타로 분류하고, FN 0건 모델도 행이 남도록 비교표 모델 목록과 합친다. 판정 불가(0·5·9·10)는 윈도우가 없어 어떤 모델의 FN 목록에도 나오지 않으므로 **분모 밖 미탐**으로 별도 표기한다.
- 점검: error_cases 기반 미탐 수와 metrics의 `n_out_seg − n_detected`(시드 평균) 합을 비교해 불일치(시드 간 차이)를 표시한다.

In [ ]:
FNL = fn_long()
FNL.to_csv(RES / "fn_by_model.csv", index=False)
key = ["notebook", "model", "split", "win_s"]
base = CMP[key].copy()                                                   # FN 0건 모델 포함
fn_set = FNL[FNL.n_miss > 0].groupby(key).apply(lambda g: {int(i): int(n) for i, n in zip(g.seg_id, g.n_miss)}).rename("missed").reset_index()
FNS = base.merge(fn_set, on=key, how="left")
FNS["missed"] = FNS.missed.apply(lambda d: d if isinstance(d, dict) else {})
for cat in ("전 채널 조용", "기타"):
    FNS[f"n_{cat}"] = FNS.missed.apply(lambda d: sum(1 for i in d if fn_category(i) == cat))
FNS["판정 불가(분모 밖)"] = len(UNDECIDABLE)
FNS["missed"] = FNS.missed.apply(lambda d: ", ".join(f"{i}×{n}" for i, n in sorted(d.items())))   # seg_id×미탐 fold 수
FNS.to_csv(RES / "fn_category_summary.csv", index=False)
print("[group_kfold_seg, 1초] 모델별 미탐 seg_id (×미탐 fold 수, 21·24~26 첫 시드)")
print(FNS[(FNS.split == "group_kfold_seg") & (FNS.win_s == 1)].drop(columns=["split", "win_s"]).to_string(index=False))
print("\n[time_block] 미탐이 있는 모델 (seg_id×미탐 fold 수, 최대 4)")
print(FNS[(FNS.split == "time_block") & (FNS.missed != "")].drop(columns="split").to_string(index=False))
print("\n범주별 미탐 (모델×세그먼트 행 수):", FNL[FNL.n_miss > 0].groupby(["split", "fn_category"]).size().to_dict())

In [ ]:
# 점검: error_cases(첫 시드) 미탐 합 vs metrics(시드 평균) n_out_seg − n_detected 합
per_fold = M_ALL[M_ALL.fold != "mean"]
m_miss = (per_fold.n_out_seg - per_fold.n_detected).groupby([per_fold.notebook, per_fold.model, per_fold.split, per_fold.win_s]).sum()
m_miss.index.names = key
e_miss = FNL.groupby(key).n_miss.sum()
CHK = pd.concat({"metrics_miss_seedmean": m_miss, "error_cases_miss_seed0": e_miss}, axis=1).fillna(0)
CHK = CHK[CHK.metrics_miss_seedmean > 0].reset_index()
CHK["diff"] = CHK.metrics_miss_seedmean - CHK.error_cases_miss_seed0
CHK["ec_covered"] = ~CHK.notebook.str.startswith("21_") | ((CHK.split == "group_kfold_seg") & (CHK.win_s == 1))   # 21 error_cases는 gkf 1초만
CHK.to_csv(RES / "fn_check_vs_metrics.csv", index=False)
print(CHK.round(3).to_string(index=False))
bad = CHK[CHK.ec_covered & (CHK["diff"].abs() > 1e-9)]
print("error_cases가 다루는 범위의 불일치:", len(bad), "행 — 시드 1·2에서만 생긴 미탐(또는 첫 시드만의 미탐)")
print(bad.round(3).to_string(index=False))

In [ ]:
# 미탐 매트릭스 그림: 모델(행) × 평가 가능 이상 세그먼트(열), group_kfold_seg 1초
f1 = FNL[(FNL.split == "group_kfold_seg") & (FNL.win_s == 1)]
mods = CMP[(CMP.split == "group_kfold_seg") & (CMP.win_s == 1)].model.tolist()
segs_eval = [i for i in range(21) if i not in UNDECIDABLE]
MAT = f1.pivot_table(index="model", columns="seg_id", values="n_miss", aggfunc="sum").reindex(index=mods, columns=segs_eval).fillna(0)
fig, ax = plt.subplots(figsize=(9, 0.28 * len(mods) + 1.5))
ax.imshow(MAT.to_numpy() > 0, cmap="Greys", vmin=0, vmax=1.4, aspect="auto")
ax.set_xticks(range(len(segs_eval))); ax.set_xticklabels(segs_eval, fontsize=8)
ax.set_yticks(range(len(mods))); ax.set_yticklabels(mods, fontsize=7)
for j, i in enumerate(segs_eval):
    if i in QUIET: ax.get_xticklabels()[j].set_color("#d62728")
ax.set_xlabel("이상 seg_id (빨강 = 전 채널 조용 3·19·20, 판정 불가 0·5·9·10 제외)")
ax.set_title("미탐(검정) 매트릭스 — group_kfold_seg, 1초")
paths.save_fig(FIG, "fn_matrix_gkf_1s.png")

## C. FP 집중 조건 — 운전 상태 · 시간 블록 · 04 정상 의심 세그먼트

- 목적: 심사 3·4번. 오경보가 저부하(state 1)에 몰린다는 가설(착수 가이드 §6)을 확인하고, 반복 오경보 세그먼트가 04 `normal_suspects.csv`(진동 상단 꼬리, 26개 중 25개 고부하)와 겹치는지 본다.
- 방법: 24~26 `error_cases.csv`의 FP 행(첫 시드, 초과 윈도우가 1개 이상인 정상 세그먼트)을 group_kfold_seg · 1초로 한정(각 정상 세그먼트가 test에 정확히 1번 나옴). 분모 = 1초 윈도우가 있는(len ≥ 10) 정상 세그먼트 수를 상태·블록별로 센다. 상태 편중은 모델별 Fisher 정확 검정(FP 여부 × 상태) + BH 보정(`stats.bh_adjust`). 세그먼트 속성은 `seg_attrs`(parquet 없으면 03 표).
- parquet 의존: 21 규칙은 `error_cases.csv`에 FP 목록이 없어 C-3에서 parquet으로 `rule1_rms_ai0` FP만 재계산한다(21 노트북에 FP 목록 저장을 추가하는 작은 PR이 더 깔끔함).

In [ ]:
W_ALL = pd.read_parquet(PQ) if HAS_PQ else None
W1 = W_ALL[W_ALL.win_s == 1.0].reset_index(drop=True) if HAS_PQ else None
SEGA = seg_attrs(W1)
SUS = pd.read_csv(paths.RESULTS / "04_data_profile_CHS" / "normal_suspects.csv")
chk = SEGA.set_index("seg_uid").loc[SUS.seg_uid, "block"].to_numpy() == SUS.block.to_numpy()
print("block 재현 = 04 normal_suspects.block:", bool(chk.all()), "| state 0 = 고부하 확인(04 의심 25/26 state0):", SUS.state.value_counts().to_dict())
NRM1 = SEGA[(SEGA.src == "normal") & (SEGA.len >= 10)]                   # 1초 윈도우가 있는 정상 세그먼트(분모)
print("분모 정상 세그먼트:", len(NRM1), "| 상태 비율:", NRM1.state_name.value_counts(normalize=True).round(4).to_dict())

E = pd.concat([pd.read_csv(p).assign(notebook=p.parent.name) for p in sorted(paths.RESULTS.glob("2*/error_cases.csv"))
               if "type" in pd.read_csv(p, nrows=0).columns], ignore_index=True)
FP = E[E.type == "FP"].merge(CMP[["model", "win_s"]].drop_duplicates(), left_on="model_id", right_on="model")
FP1 = FP[(FP.split == "group_kfold_seg") & (FP.win_s == 1)].drop_duplicates(["model_id", "seg_uid"])
FP1 = FP1.drop(columns=["state"]).merge(SEGA[["seg_uid", "state", "state_name", "block"]], on="seg_uid", how="left")
print("FP 행(gkf, 1초):", len(FP1), "모델", FP1.model_id.nunique())

In [ ]:
# C-1 상태별 FP율 + Fisher 검정
n_state = NRM1.state_name.value_counts()
rows = []
for m, g in FP1.groupby("model_id", sort=False):
    fp = g.state_name.value_counts().reindex(["고부하", "저부하"], fill_value=0)
    tab = [[fp["고부하"], n_state["고부하"] - fp["고부하"]], [fp["저부하"], n_state["저부하"] - fp["저부하"]]]
    orr, p = sst.fisher_exact(tab)
    rows.append({"model": m, "notebook": g.notebook.iloc[0], "fp_seg_high": fp["고부하"], "fp_seg_low": fp["저부하"],
                 "fpr_seg_high": fp["고부하"] / n_state["고부하"], "fpr_seg_low": fp["저부하"] / n_state["저부하"],
                 "odds_ratio_high_vs_low": orr, "fisher_p": p})
FPS = pd.DataFrame(rows)
FPS["p_adj_bh"] = stats.bh_adjust(FPS.fisher_p.to_numpy())
FPS["concentrated_in"] = np.where(FPS.p_adj_bh >= 0.05, "차이 없음", np.where(FPS.fpr_seg_high > FPS.fpr_seg_low, "고부하", "저부하"))
FPS.to_csv(RES / "fp_by_state.csv", index=False)
print(FPS.round(4).to_string(index=False))
print("\n집중 방향 모델 수:", FPS.concentrated_in.value_counts().to_dict())

In [ ]:
# C-2 블록별 FP율 · 04 정상 의심 세그먼트와의 겹침 · 모델 공통 FP 세그먼트
n_block = NRM1.block.value_counts().sort_index()
FPB = (FP1.groupby(["model_id", "block"]).size().unstack(fill_value=0).reindex(columns=n_block.index, fill_value=0) / n_block).round(4)
FPB.columns = [f"fpr_seg_block{int(b)}" for b in FPB.columns]
FPB.to_csv(RES / "fp_by_block.csv")
print(FPB.to_string())
COM = (FP1.groupby("seg_uid").agg(n_models=("model_id", "nunique"), models=("model_id", lambda s: ",".join(sorted(s))))
       .join(SEGA.set_index("seg_uid")[["seg_id", "len", "state_name", "block"]])
       .join(SUS.set_index("seg_uid")[["criteria"]]).sort_values("n_models", ascending=False))
COM["is_suspect_04"] = COM.criteria.notna()
COM.to_csv(RES / "fp_common_segments.csv")
nm = FP1.model_id.nunique()
print(f"\n모델 {nm}개 중 과반이 FP로 낸 세그먼트:")
print(COM[COM.n_models > nm / 2].drop(columns="models").to_string())
ov = FP1.assign(sus=FP1.seg_uid.isin(SUS.seg_uid)).groupby("model_id").sus.agg(["sum", "size", "mean"]).round(3)
print("\n모델별 FP 세그먼트 중 04 의심 비율 (의심 26 / 분모", len(NRM1), "=", round(SUS.seg_uid.isin(NRM1.seg_uid).sum() / len(NRM1), 3), ")")
print(ov.to_string())

In [ ]:
# C-3 (parquet 필요) 21 rule1_rms_ai0 FP 재계산 — 21 error_cases.csv에 FP 목록이 없음
if HAS_PQ:
    rows = []
    for k in range(5):
        tr = W1[(W1.fold != k) & (W1.label == 0)]; te = W1[(W1.fold == k) & (W1.label == 0)]
        thr = evaluate.threshold_from_normal(tr.AI0_Vibration_rms)
        ex = te.assign(exceed=te.AI0_Vibration_rms > thr).groupby("seg_uid").exceed.any()
        rows += [{"model_id": "rule1_rms_ai0", "seg_uid": u} for u in ex[ex].index]
    R1 = pd.DataFrame(rows).merge(SEGA[["seg_uid", "state_name", "block"]], on="seg_uid")
    print("rule1 FP 세그먼트:", len(R1), "| 상태:", R1.state_name.value_counts().to_dict(), "| 04 의심 겹침:", int(R1.seg_uid.isin(SUS.seg_uid).sum()))
else:
    print("parquet 없음 — C-3 건너뜀")

## D. 정상 vs 이상 상관 구조 변화 (parquet 필요)

- 목적: 심사 3·5번. 크기(RMS)가 아니라 **채널·피처 간 관계**가 깨지는지 본다. 조용한 이상(3·19·20)을 잡을 수 있는 피처 조합 후보를 찾는다.
- 방법: 1초 윈도우의 정상 전체 vs 이상 전체 Spearman 상관(`corr_break`), 쌍별 차이 ρ_이상 − ρ_정상 상위 쌍, 차이 히트맵. 12에서 확인한 완전 중복 열(형상 kurt·skew = 진폭 kurt·skew)은 제외한다(`results/12_feature_statistics_CHS/duplicate_columns.csv`).
- 해석 주의: 윈도우는 독립 표본이 아니고(한 세그먼트 윈도우끼리 거의 같음) 이상 윈도우는 96개·17세그먼트뿐이다. 세그먼트 브로드캐스트 열(`vib_corr01`, `cur_ac1`, `cur_fit_*`)이 낀 쌍의 상관은 사실상 세그먼트 간 상관이다. p값은 보고하지 않고 기술 통계로만 쓴다.

In [ ]:
if HAS_PQ:
    FEATS = features.feature_columns(W_ALL)
    dup = pd.read_csv(paths.RESULTS / "12_feature_statistics_CHS" / "duplicate_columns.csv")
    drop = {b if "_shape_" in b else a for a, b in zip(dup.col_a, dup.col_b)}
    F_CORR = [c for c in FEATS if c not in drop]
    CB, CN, CO = corr_break(W1[W1.label == 0], W1[W1.label == 1], F_CORR)
    CB.to_csv(RES / "corr_break_normal_vs_outlier.csv", index=False)
    print("열 수:", len(F_CORR), "| |diff| >= 0.5 쌍:", int((CB.abs_diff >= 0.5).sum()), "/", len(CB))
    print(CB.head(15).round(3).to_string(index=False))
    D = (CO - CN).loc[F_CORR, F_CORR]
    fig, ax = plt.subplots(figsize=(10, 8.5))
    im = ax.imshow(D.to_numpy(), vmin=-1.5, vmax=1.5, cmap="RdBu_r")
    ax.set_xticks(range(len(F_CORR))); ax.set_xticklabels(F_CORR, rotation=90, fontsize=6)
    ax.set_yticks(range(len(F_CORR))); ax.set_yticklabels(F_CORR, fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.7, label="rho_이상 − rho_정상 (Spearman)")
    ax.set_title("상관 구조 변화 — 1초 윈도우, 정상 vs 이상")
    paths.save_fig(FIG, "corr_break_heatmap.png")
else:
    print("parquet 없음 — D 건너뜀")

## E. 정상 부분공간 이탈 — PCA SPE(Q) · Hotelling T² (parquet 필요)

- 목적: 심사 3번. 조용한 이상 3·19·20이 "정상 PCA 부분공간 안(T² 작음)인데 잔차(SPE)만 큰" 경우인지, 둘 다 정상 범위인지(→ 라벨 노이즈 쪽 근거) 가린다.
- 방법: 피처 집합 2개(진폭 18열 `amp18`, 전체 33열 `all33`)마다 (1) **fold OOF**: fold k를 뺀 정상 윈도우로 StandardScaler+PCA(누적 95% 성분) 적합 → fold k 윈도우의 SPE·T², 정상 OOF 점수 q99를 기준선으로 이상 세그먼트별 중앙값/q99 비율 → `fn_pca_spe_t2.csv`. (2) 시각화용으로 정상 전체 적합 PC1–PC2 산점도(정상은 상태별 색, 이상 겹침, 3·19·20 주석). `all33`의 `vib_rms_ratio` NaN은 train 정상 중앙값으로 채운다(21 rule3b와 같은 처리).

In [ ]:
if HAS_PQ:
    AMP18 = [c for c in FEATS if c.startswith(("AI0_", "AI1_", "AI2_")) and "_shape_" not in c and c.rsplit("_", 1)[-1] in features.AMP_KEYS]
    assert len(AMP18) == 18 and len(FEATS) == 33
    FSETS = {"amp18": AMP18, "all33": FEATS}
    rows, OOF = [], {}
    for name, cols in FSETS.items():
        sc = pd.DataFrame(index=W1.index, columns=["spe", "t2"], dtype=float)
        for k in range(5):
            trn = W1[(W1.fold != k) & (W1.label == 0)]
            fill = trn[cols].median()
            s, n_k = pca_spe_t2(trn[cols].fillna(fill), W1.loc[W1.fold == k, cols].fillna(fill), cols)
            sc.loc[s.index] = s.to_numpy()
            rows.append({"feature_set": name, "fold": k, "n_components_95": n_k})
        OOF[name] = sc
    NCOMP = pd.DataFrame(rows); print(NCOMP.groupby("feature_set").n_components_95.agg(["min", "max"]).to_dict("index"))
    out = []
    for name, sc in OOF.items():
        d = W1[["seg_uid", "label"]].join(sc)
        q99 = d[d.label == 0][["spe", "t2"]].quantile(0.99)
        g = d[d.label == 1].groupby("seg_uid")[["spe", "t2"]].median()
        g = g.assign(feature_set=name, spe_q99_normal=q99.spe, t2_q99_normal=q99.t2,
                     spe_ratio=g.spe / q99.spe, t2_ratio=g.t2 / q99.t2).reset_index()
        out.append(g)
    PT = pd.concat(out, ignore_index=True).merge(SEGA[["seg_uid", "seg_id", "vib_grade", "cur_grade"]], on="seg_uid")
    PT["fn_category"] = PT.seg_id.map(fn_category)
    PT["region"] = np.select([(PT.spe_ratio > 1) & (PT.t2_ratio > 1), PT.spe_ratio > 1, PT.t2_ratio > 1],
                             ["SPE·T² 모두 초과", "SPE만 초과(구조 이탈)", "T²만 초과(크기 이탈)"], "둘 다 정상 범위")
    PT.to_csv(RES / "fn_pca_spe_t2.csv", index=False)
    print(PT.sort_values(["feature_set", "seg_id"])[["feature_set", "seg_id", "fn_category", "spe_ratio", "t2_ratio", "region", "vib_grade", "cur_grade"]]
          .round(3).to_string(index=False))
else:
    print("parquet 없음 — E 건너뜀")

In [ ]:
if HAS_PQ:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, (name, cols) in zip(axes, FSETS.items()):
        nrm = W1[W1.label == 0]; fill = nrm[cols].median()
        rep = stats.pca_fit_report(nrm[cols].fillna(fill), cols, n_components=2)
        Z = rep["pca"].transform(rep["scaler"].transform(W1[cols].fillna(fill).to_numpy(dtype=float)))
        for st_, col in ((0, "#eb6834"), (1, "#2a78d6")):
            m = (W1.label == 0) & (W1.state == st_)
            ax.scatter(Z[m, 0], Z[m, 1], s=6, alpha=.35, color=col, label=f"정상 {STATE_NAME[st_]}(state {st_})")
        mo = (W1.label == 1).to_numpy()
        ax.scatter(Z[mo, 0], Z[mo, 1], s=22, color="k", marker="x", label="이상 윈도우")
        for i in QUIET:
            mi = (W1.seg_uid == f"outlier_{i}").to_numpy()
            if mi.any():
                c = np.median(Z[mi], axis=0)
                ax.annotate(f"seg {i}", c, fontsize=9, color="#d62728", xytext=(6, 6), textcoords="offset points",
                            arrowprops={"arrowstyle": "-", "color": "#d62728"})
        ev = rep["explained"].explained_ratio
        ax.set_xlabel(f"PC1 ({ev.iloc[0]:.1%})"); ax.set_ylabel(f"PC2 ({ev.iloc[1]:.1%})")
        ax.set_title(f"{name}: 정상 적합 PCA 투영 (1초)"); ax.legend(fontsize=7, markerscale=2); ax.grid(alpha=.3)
    paths.save_fig(FIG, "fn_pca_scatter.png")

## F. 세그먼트 수준 KS 통계 (parquet 필요)

- 목적: 12의 `group_test_normal_vs_outlier.csv`(Welch·MWU·Cliff's δ = 위치 차이)를 보완해 **분포 형태 차이**(KS)를 본다. 위치는 같고 퍼짐만 다른 피처를 찾는다.
- 방법: 1초 윈도우를 `stats.segment_representatives`로 세그먼트 중앙값 1행(정상 530 / 이상 17)으로 줄이고 피처별 `scipy.stats.ks_2samp`, BH 보정, 12의 Cliff's δ와 나란히 둔다. `ks_rank`와 `|cliff|` 순위가 크게 다른 피처가 "형태 차이" 후보다.

In [ ]:
if HAS_PQ:
    SR = stats.segment_representatives(W1, FEATS)
    rows = []
    for c in FEATS:
        a = SR.loc[SR.label == 0, c].dropna(); b = SR.loc[SR.label == 1, c].dropna()
        r = sst.ks_2samp(a, b)
        rows.append({"feature": c, "n_normal": len(a), "n_outlier": len(b), "ks_stat": r.statistic, "ks_p": r.pvalue})
    KS = pd.DataFrame(rows)
    KS["p_adj_bh"] = stats.bh_adjust(KS.ks_p.to_numpy())
    g12 = paths.RESULTS / "12_feature_statistics_CHS" / "group_test_normal_vs_outlier.csv"
    if g12.exists():
        KS = KS.merge(pd.read_csv(g12)[["feature", "cliff_delta", "group"]], on="feature", how="left")
        KS["ks_rank"] = KS.ks_stat.rank(ascending=False); KS["cliff_rank"] = KS.cliff_delta.abs().rank(ascending=False)
    KS = KS.sort_values("ks_stat", ascending=False).reset_index(drop=True)
    KS.to_csv(RES / "ks_segment_normal_vs_outlier.csv", index=False)
    print(KS.round(4).to_string(index=False))
else:
    print("parquet 없음 — F 건너뜀")

## 결론

| 절 | 발견 | 시사점 | 활용 아이디어 |
|---|---|---|---|
| A 비교표 | (실행 후 기입) | (실행 후 기입) | (실행 후 기입) |
| B FN 이원화 | (실행 후 기입) | (실행 후 기입) | (실행 후 기입) |
| C FP 집중 | (실행 후 기입) | (실행 후 기입) | (실행 후 기입) |
| D 상관 구조 | (실행 후 기입) | (실행 후 기입) | (실행 후 기입) |
| E PCA SPE·T² | (실행 후 기입) | (실행 후 기입) | (실행 후 기입) |
| F KS | (실행 후 기입) | (실행 후 기입) | (실행 후 기입) |

- 한계: 이상 이벤트 1건(평가 가능 세그먼트 17개), 24~26 error_cases는 첫 시드만, 21 규칙 FP 목록 부재(C-3 재계산), 상관·PCA는 윈도우 비독립. (실행 후 보완)
- 다음 단계: (실행 후 기입)